In [9]:
# Create sample IFCT database
ifct_data = """food_name,calories,protein,carbs,fat,fiber
rice,130,2.7,28.0,0.3,0.4
brown rice,216,4.5,44.8,1.8,3.5
roti,120,3.5,24.0,0.4,2.1
chapati,120,3.5,24.0,0.4,2.1
dal,116,7.6,20.0,0.4,3.8
rajma,144,8.7,26.0,0.5,6.4
biryani,163,5.2,25.0,4.1,1.2
idli,58,1.9,11.0,0.1,0.5
dosa,168,3.4,25.0,6.4,1.2
sambar,42,2.3,7.0,0.6,2.1
paneer,265,18.3,1.2,20.8,0.0
chicken curry,243,28.0,8.0,11.0,1.2
egg,155,13.0,1.1,11.0,0.0
bread,265,9.0,49.0,3.2,2.7
banana,89,1.1,23.0,0.3,2.6
apple,52,0.3,14.0,0.2,2.4
milk,42,3.4,5.0,1.0,0.0
curd,98,11.0,3.4,4.3,0.0
poha,333,7.0,76.0,0.5,2.7
upma,145,3.9,24.0,4.2,2.1
puri,326,7.0,45.0,14.0,2.1
aloo sabzi,98,2.1,18.0,2.8,2.4
palak paneer,187,9.8,8.3,13.0,2.6
butter chicken,164,14.0,5.0,10.0,0.8
naan,317,10.0,55.0,6.0,2.0"""

# Save as CSV file
with open("ifct_sample.csv", "w") as f:
    f.write(ifct_data)

print("✅ IFCT database created successfully!")

✅ IFCT database created successfully!


In [10]:
# Install all required libraries
!pip install transformers torch gradio Pillow -q

print("✅ All libraries installed!")

✅ All libraries installed!


In [11]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

print("⏳ Loading BLIP model... (first time takes 2-3 mins)")

# Load model - using smaller base version for speed
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"✅ BLIP model loaded! Running on: {device}")

⏳ Loading BLIP model... (first time takes 2-3 mins)


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

✅ BLIP model loaded! Running on: cpu


In [12]:
import pandas as pd
import requests

# Load IFCT database
df = pd.read_csv("ifct_sample.csv")

def search_ifct(food_name):
    """Search IFCT database for nutrition info"""
    # Try exact match first
    result = df[df['food_name'].str.lower() == food_name.lower()]

    # If not found try partial match
    if result.empty:
        result = df[df['food_name'].str.contains(food_name.lower(), case=False, na=False)]

    if not result.empty:
        data = result.iloc[0]
        return {
            "source": "IFCT",
            "food": data['food_name'],
            "calories": data['calories'],
            "protein": data['protein'],
            "carbs": data['carbs'],
            "fat": data['fat'],
            "fiber": data['fiber']
        }
    return None

def search_usda(food_name):
    """Search USDA API as fallback"""
    try:
        api_key = "DEMO_KEY"  # Free demo key
        url = f"https://api.nal.usda.gov/fdc/v1/foods/search"
        params = {
            "query": food_name,
            "api_key": api_key,
            "pageSize": 1
        }
        response = requests.get(url, params=params, timeout=5)
        data = response.json()

        if data.get('foods'):
            food = data['foods'][0]
            nutrients = {n['nutrientName']: n['value']
                        for n in food.get('foodNutrients', [])}
            return {
                "source": "USDA",
                "food": food['description'],
                "calories": nutrients.get('Energy', 0),
                "protein": nutrients.get('Protein', 0),
                "carbs": nutrients.get('Carbohydrate, by difference', 0),
                "fat": nutrients.get('Total lipid (fat)', 0),
                "fiber": nutrients.get('Fiber, total dietary', 0)
            }
    except:
        pass
    return None

def get_nutrition(food_name):
    """Main function - IFCT first, USDA fallback"""
    # Clean food name
    food_name = food_name.strip().lower()

    # Try IFCT first
    result = search_ifct(food_name)
    if result:
        print(f"✅ Found in IFCT!")
        return result

    # Fallback to USDA
    print(f"⚠️ Not in IFCT, trying USDA...")
    result = search_usda(food_name)
    if result:
        print(f"✅ Found in USDA!")
        return result

    return None

print("✅ Nutrition lookup ready!")

✅ Nutrition lookup ready!


In [13]:
def identify_food(image):
    """Use BLIP to identify food in image"""
    # Prepare image
    inputs = processor(
        image,
        text="what food is in this image?",
        return_tensors="pt"
    ).to(device)

    # Generate description
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        num_beams=4
    )

    # Decode result
    description = processor.decode(output[0], skip_special_tokens=True)

    # Extract main food name (first 1-2 words usually)
    food_name = description.split(',')[0].strip()
    food_name = food_name.replace("a plate of", "").strip()
    food_name = food_name.replace("a bowl of", "").strip()
    food_name = food_name.replace("a dish of", "").strip()

    return food_name, description

print("✅ Food identifier ready!")

✅ Food identifier ready!


In [14]:
import gradio as gr

def analyze_meal(image):
    """Main pipeline - image to nutrition"""

    if image is None:
        return "❌ Please upload an image!"

    # Step 1 - Identify food
    print("🔍 Identifying food...")
    food_name, full_description = identify_food(image)
    print(f"Detected: {full_description}")

    # Step 2 - Get nutrition
    print(f"📊 Looking up nutrition for: {food_name}")
    nutrition = get_nutrition(food_name)

    # Step 3 - Format output
    if nutrition:
        result = f"""
🍽️ MEAL ANALYSIS RESULTS
{'='*35}

📌 Detected Food: {full_description}
🔍 Searched For: {nutrition['food']}
📚 Data Source: {nutrition['source']}

{'='*35}
NUTRITION (per 100g)
{'='*35}
🔥 Calories:  {nutrition['calories']} kcal
💪 Protein:   {nutrition['protein']} g
🍞 Carbs:     {nutrition['carbs']} g
🧈 Fat:       {nutrition['fat']} g
🌾 Fiber:     {nutrition['fiber']} g
{'='*35}
        """
    else:
        result = f"""
🍽️ MEAL ANALYSIS RESULTS
{'='*35}
📌 Detected: {full_description}
❌ Nutrition data not found in IFCT or USDA
💡 Try: Describe the food in text below
        """

    return result

# Build Gradio interface
demo = gr.Interface(
    fn=analyze_meal,
    inputs=gr.Image(
        type="pil",
        label="📸 Upload Your Meal Photo"
    ),
    outputs=gr.Textbox(
        label="📊 Nutrition Results",
        lines=15
    ),
    title="🍽️ AI Meal Nutrition Analyzer",
    description="Upload a photo of your meal to get nutritional information!",
    examples=[],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d137a1306c39c2b257.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
# Run this cell again
demo.close()  # Close old session
demo.launch(share=True)  # Relaunch fresh


Closing server running on port: 7861
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d137a1306c39c2b257.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
